# Radar Cidadão — NLP / Colab v8.2

**Notebook organizado para a correção v8.2.** Usa exclusivamente o `geracao.py` publicado no GitHub (`b82d48f`), sem sobrescrevê-lo. Não apaga os 278 arquivos de cache existentes.

**Como usar:** abra este arquivo no Google Colab, selecione **Ambiente de execução → Alterar tipo de ambiente de execução → GPU T4** (ou outra GPU adequada) e execute as células **1, 2 e 3** na ordem. Em seguida execute **4 (juiz)** e **5 (amostra)**, examine `avaliacao_v8_2.csv` no Drive e **só depois** execute as células opcionais **6 a 8**.

A geração do modelo exige GPU e internet. A etapa 5 **não mede aprovação humana automaticamente**; a meta de 90% depende da marcação S/N por pessoas. Nenhum dos resultados da v8 é sobrescrito.

In [ ]:
# 1. Conectar Drive, obter a versão correta do GitHub e instalar dependências
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import subprocess, os, sys

REPO = Path("/content/radar-cidadao")
PROJETO = REPO / "nlp-fabrica-corpus"
URL = "https://github.com/sarahmfurlan-sudo/radar-cidadao.git"
COMMIT_ESPERADO = "b82d48f"

def git(*args):
    return subprocess.run(["git", "-C", str(REPO), *args],
                          check=True, text=True, capture_output=True).stdout.strip()

if not (REPO / ".git").is_dir():
    if REPO.exists():
        raise RuntimeError("A pasta do projeto existe mas não é um repositório. Não vou sobrescrever.")
    subprocess.run(["git", "clone", URL, str(REPO)], check=True)
else:
    if git("status", "--porcelain"):
        raise RuntimeError("Há alterações locais no Colab. Não vou sobrescrever o código.")
    git("fetch", "origin")
    git("merge", "--ff-only", "origin/main")

commit = git("rev-parse", "--short", "HEAD")
print("Commit:", commit)
if commit != COMMIT_ESPERADO:
    raise RuntimeError(f"Commit diferente do validado: {commit}. Esperado: {COMMIT_ESPERADO}.")
os.chdir(PROJETO)
SRC = str(PROJETO / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

# Instala as dependências declaradas no projeto (inclui pypdf).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
# O spaCy exige baixar o modelo em uma sessão nova.
import spacy
try:
    spacy.load("pt_core_news_md")
    print("Modelo spaCy de português já instalado.")
except OSError:
    subprocess.run([sys.executable, "-m", "spacy", "download", "pt_core_news_md"], check=True)
    spacy.load("pt_core_news_md")
    print("Modelo spaCy de português instalado.")

import torch
if not torch.cuda.is_available():
    print("AVISO: sem GPU. Ative GPU nas configurações do Colab antes de carregar o Qwen.")
else:
    print("GPU:", torch.cuda.get_device_name(0))

Mounted at /content/drive
Commit: b82d48f
Modelo spaCy de português já instalado.
GPU: Tesla T4


In [ ]:
# 2. Ingestão, limpeza, anonimização e segmentação — somente leitura da base
import logging, time
logging.disable(logging.WARNING)
from fabrica_corpus.ingestao import ingerir_pasta
from fabrica_corpus.limpeza import limpar_corpus
from fabrica_corpus.anonimizacao import anonimizar_corpus
from fabrica_corpus.segmentacao import segmentar

limpos, estat_limpeza = limpar_corpus(ingerir_pasta("dados/entrada"))
anon, estat_anon = anonimizar_corpus(limpos)
segs, estat_seg = segmentar(anon)
print("Segmentos:", len(segs), "| média de tokens:", estat_seg.get("tokens_media"))
print("Anonimização:", estat_anon)
if len(segs) != 306:
    print("ATENÇÃO: a referência anterior tinha 306 segmentos. Confira a base antes de comparar os resultados.")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Segmentos: 306 | média de tokens: 482.9
Anonimização: {'cpf': 0, 'email': 0, 'telefone': 0, 'nome': 127}


In [ ]:
# 3. Carregar módulo v8.2 e Qwen2.5-14B — GPU necessária
import importlib
from fabrica_corpus import geracao
geracao = importlib.reload(geracao)
print("Módulo:", geracao.__file__)
print("Versão:", geracao.VERSAO_PROMPT, "| Modelo:", geracao.MODELO_GERADOR,
      "| Fatos:", geracao.USAR_FATOS, "| Juiz:", geracao.USAR_JUIZ)
assert geracao.VERSAO_PROMPT == "v8_2", "O Colab não está usando o módulo correto."
assert str(geracao.__file__).startswith(str(PROJETO)), "Módulo de fora do repositório!"
assert torch.cuda.is_available(), "Ative uma GPU antes de carregar o Qwen."

# Carrega apenas uma vez por sessão. Não substitui o geracao.py.
if "gerar" not in globals() or not callable(gerar):
    gerar = geracao.carregar_qwen()
CACHE = Path("/content/drive/MyDrive/radar-cidadao/cache_geracao")
CACHE.mkdir(parents=True, exist_ok=True)
print("Cache:", CACHE, "| arquivos:", len(list(CACHE.glob("*.json"))))
print("Pronto.")

Módulo: /content/radar-cidadao/nlp-fabrica-corpus/src/fabrica_corpus/geracao.py
Versão: v8_2 | Modelo: Qwen/Qwen2.5-14B-Instruct | Fatos: True | Juiz: True


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Cache: /content/drive/MyDrive/radar-cidadao/cache_geracao | arquivos: 288
Pronto.


In [ ]:
# 4. Medir a concordância do juiz com os 28 rótulos antigos
inicio = time.time()
r = geracao.medir_juiz("revisao/rotulos_revisao_7b.csv", segs, gerar)
print(f"Tempo: {(time.time()-inicio)/60:.1f} min")
print(f"Rótulos no CSV: {r['itens_csv']} | avaliados: {r['itens']} | não avaliados: {r['itens_nao_avaliados']}")
print(f"Contexto expandido: {r['itens_contexto_expandido']} | concordância: {r['concordancia']:.0%}")
for chave in ("humano_S_sistema_aprova", "humano_S_sistema_reprova",
              "humano_N_sistema_aprova", "humano_N_sistema_reprova"):
    print(chave, ":", r[chave])
print("Divergências:")
for item, motivo in r["divergencias"]:
    print(" ", item, "->", motivo)
# Contexto expandido é uma aproximação: não equivale a identificar o segmento original.

Tempo: 5.5 min
Rótulos no CSV: 28 | avaliados: 28 | não avaliados: 0
Contexto expandido: 4 | concordância: 61%
humano_S_sistema_aprova : 12
humano_S_sistema_reprova : 8
humano_N_sistema_aprova : 3
humano_N_sistema_reprova : 5
Divergências:
  qa_0016 -> humano S; sistema: juiz (mesmo_agente): A pergunta menciona 'a prefeitura', enquanto a resposta descreve uma ação atribuída genericamente aos órgãos competentes.
  qa_0041 -> humano S; sistema: juiz (apoiada, responde): A resposta menciona ações específicas da Defesa Civil que não estão diretamente citadas no trecho fornecido, além de não responder exatamente à pergunta feita.
  qa_0066 -> humano S; sistema: juiz (apoiada, responde, mesmo_agente): A resposta mistura instruções sobre o que fazer em caso de acidente com animais peçonhentos com as instruções específicas sobre como lidar com animais mortos devido à inundação, além de incluir informações sobre ações que não são solicitadas na pergunta.
  qa_0074 -> humano S; sistema: juiz (ap

In [ ]:
# 5. Gerar e salvar amostra fixa da v8.2 (sem mexer no arquivo da v8)
import csv
aval = geracao.conjunto_avaliacao(segs)
print("Segmentos fixos:", [s.id for s in aval])
inicio = time.time()
qa_av, e1 = geracao.gerar_qa(aval, gerar, total=24, pasta_cache=CACHE)
chat_av, e2 = geracao.gerar_chat(aval, gerar, total=4, pasta_cache=CACHE)
rec_av, e3 = geracao.gerar_recusas(gerar, pasta_cache=CACHE)
todos_chats = chat_av + rec_av
print(f"Tempo: {(time.time()-inicio)/60:.1f} min")
print("QA:", e1, "\nDiálogos:", e2, "\nRecusas:", e3)

SAIDA_AV = Path("/content/drive/MyDrive/radar-cidadao/avaliacao_v8_2.csv")
# Não sobrescrever uma planilha que já tenha sido revisada por pessoas.
if SAIDA_AV.exists():
    print("ARQUIVO JÁ EXISTE: não vou sobrescrever", SAIDA_AV)
else:
    with SAIDA_AV.open("w", newline="", encoding="utf-8-sig") as f:
        w = csv.writer(f, delimiter=";")
        w.writerow(["id","segmento_id","fonte","paginas","pergunta / diálogo",
                    "resposta","ok? (S/N)","observação"])
        for e in qa_av:
            w.writerow([e["id"],e["segmento_id"],e["fonte"],e["paginas"],
                        e["pergunta"],e["resposta"],"",""])
        for e in todos_chats:
            texto = "\n".join(f"{m['role']}: {m['content']}" for m in e["mensagens"][1:])
            w.writerow([e["id"],e.get("segmento_id"),e["fonte"],e["paginas"],
                        texto,"","",""])
    print("Planilha criada:", SAIDA_AV)
print(f"Gerados: {len(qa_av)}/24 perguntas; {len(chat_av)}/4 diálogos; {len(rec_av)} recusas.")
print("ATENÇÃO: qualidade >= 90% só pode ser verificada após a revisão humana da planilha.")

Segmentos fixos: ['mcid_2024_periferia_sem_risco_0003', 'mcid_2026_mapeamento_riscos_0071', 'ms_2017_cartilha_alerta_chuvas_0011', 'ms_2025_plano_contingencia_chuvas_0027', 'ms_2026_enchentes_0001', 'mcid_2024_periferia_sem_risco_0002', 'mcid_2026_mapeamento_riscos_0074', 'ms_2017_cartilha_alerta_chuvas_0001', 'ms_2025_plano_contingencia_chuvas_0039', 'ms_2026_enchentes_0002', 'mcid_2024_periferia_sem_risco_0001', 'mcid_2026_mapeamento_riscos_0048']
[perguntas] 2/24 | segmento mcid_2024_periferia_sem_risco_0003 | 2 aprovadas, 2 tentativa(s)
[perguntas] 3/24 | segmento mcid_2026_mapeamento_riscos_0071 | 1 aprovadas, 3 tentativa(s)
[perguntas] 5/24 | segmento ms_2017_cartilha_alerta_chuvas_0011 | 2 aprovadas, 2 tentativa(s)
[perguntas] 7/24 | segmento ms_2025_plano_contingencia_chuvas_0027 | 2 aprovadas, 3 tentativa(s)
[perguntas] 9/24 | segmento ms_2026_enchentes_0001 | 2 aprovadas, 2 tentativa(s)
[perguntas] 11/24 | segmento mcid_2024_periferia_sem_risco_0002 | 2 aprovadas, 3 tentativa

In [ ]:
import json

tema = geracao.TEMAS_RECUSA[0]

prompt = geracao.PROMPT_RECUSA.format(
    persona_assistente=geracao.PERSONA_GERACAO,
    persona_cidadao=geracao.PERSONA_CIDADAO_GERACAO,
    pedido=tema["pedido"],
    destino=tema["destino"]
)

saida = gerar([{"role": "user", "content": prompt}])

print("=== RESPOSTA ORIGINAL DO QWEN ===")
print(saida)

print("\n=== JSON INTERPRETADO ===")
try:
    mensagens = geracao._extrair_json(saida, "mensagens")["mensagens"]
    print(json.dumps(mensagens, ensure_ascii=False, indent=2))
    print("\n=== VALIDACAO ===")
    print(geracao._conferir_recusa(mensagens, tema))
except Exception as erro:
    print(type(erro).__name__, str(erro))

=== RESPOSTA ORIGINAL DO QWEN ===
{
  "mensagens": [
    {
      "papel": "cidadao",
      "texto": "Tenho febre e dor no corpo depois de andar na enchente. É leptospirose?"
    },
    {
      "papel": "assistente",
      "texto": "Não posso diagnosticar. Procure uma unidade de saúde ou um médico."
    }
  ]
}

=== JSON INTERPRETADO ===
[
  {
    "papel": "cidadao",
    "texto": "Tenho febre e dor no corpo depois de andar na enchente. É leptospirose?"
  },
  {
    "papel": "assistente",
    "texto": "Não posso diagnosticar. Procure uma unidade de saúde ou um médico."
  }
]

=== VALIDACAO ===
(False, 'diálogo sem alternância cidadão/assistente ou curto demais')


In [ ]:
import subprocess
import importlib

# Atualizar o código sem reiniciar o Colab
subprocess.run(
    ["git", "-C", "/content/radar-cidadao", "pull", "--ff-only"],
    check=True
)

import fabrica_corpus.geracao as geracao
geracao = importlib.reload(geracao)

# Confirmar que a correção foi carregada
assert hasattr(geracao, "_completar_recusa_curta")

# Gerar somente as recusas
recusas, estat = geracao.gerar_recusas(
    gerar,
    total=12,
    pasta_cache=CACHE
)

print("Recusas geradas:", len(recusas), "/12")
print("Estatísticas:", estat)

[recusas] diagnóstico | ok
[recusas] remédio | ok
[recusas] picada | ok
[recusas] estrutura da casa | ok
[recusas] emergência agora | ok
[recusas] previsão exata | ok
[recusas] processo judicial | ok
[recusas] auxílio em dinheiro | pulado: fala do trecho como se o leitor visse o documento
[recusas] seguro | ok
[recusas] obra pública | ok
[recusas] dados de terceiros | ok
[recusas] fora do assunto | ok
Recusas geradas: 11 /12
Estatísticas: {'exemplos': 11, 'temas_pulados': ['auxílio em dinheiro']}


In [ ]:
import json
from pathlib import Path

arquivo = Path("/content/drive/MyDrive/radar-cidadao/recusas_v8_2.json")

with open(arquivo, "w", encoding="utf-8") as f:
    json.dump(recusas, f, ensure_ascii=False, indent=2)

print("Salvas:", len(recusas), "recusas")
print("Arquivo:", arquivo)

Salvas: 11 recusas
Arquivo: /content/drive/MyDrive/radar-cidadao/recusas_v8_2.json


In [17]:
# Execute depois de o instalador concluir o push. Mantenha a GPU/Qwen carregados.
import subprocess, importlib, csv
from pathlib import Path
repo = '/content/radar-cidadao'
subprocess.run(['git', '-C', repo, 'pull', '--ff-only'], check=True)
import fabrica_corpus.geracao as geracao
geracao = importlib.reload(geracao)
assert geracao.VERSAO_QA_CHAT == 'v8_3'
assert geracao.VERSAO_PROMPT == 'v8_2'  # recusas preservadas
assert 'gerar' in globals() and 'segs' in globals() and 'CACHE' in globals()
aval = geracao.conjunto_avaliacao(segs)
qa, est_qa = geracao.gerar_qa(aval, gerar, total=24, pasta_cache=CACHE)
chat, est_chat = geracao.gerar_chat(aval, gerar, total=4, pasta_cache=CACHE)
print('QA:', len(qa), '/24', est_qa)
print('Diálogos:', len(chat), '/4', est_chat)
# Arquivo novo para não sobrescrever a revisão da v8.2
saida = Path('/content/drive/MyDrive/radar-cidadao/avaliacao_v8_3.csv')
with saida.open('w', encoding='utf-8-sig', newline='') as f:
    w=csv.writer(f,delimiter=';')
    w.writerow(['id','segmento_id','fonte','paginas','pergunta / diálogo','resposta','ok? (S/N)','observação'])
    for x in qa:
        w.writerow([x['id'],x['segmento_id'],x['fonte'],x['paginas'],x['pergunta'],x['resposta'],'',''])
    for x in chat:
        falas = '\n'.join(m['role'] + ': ' + m['content'] for m in x['mensagens'] if m['role'] != 'system')
        w.writerow([x['id'],x['segmento_id'],x['fonte'],x['paginas'],falas,'','',''])
print('Revisar:', saida)
print('Recusas anteriores NÃO foram alteradas. Não execute a geração completa ainda.')

[perguntas] 2/24 | segmento mcid_2024_periferia_sem_risco_0003 | 2 aprovadas, 1 tentativa(s)


KeyboardInterrupt: 

In [18]:
from pathlib import Path

p = Path("/content/drive/MyDrive/radar-cidadao/avaliacao_v8_3.csv")

print("Arquivo existe:", p.exists())
if p.exists():
    print("Tamanho:", p.stat().st_size, "bytes")
    print("Linhas:", sum(1 for _ in p.open(encoding="utf-8-sig")) - 1)

Arquivo existe: True
Tamanho: 6672 bytes
Linhas: 74


---

### Somente depois de revisar a amostra

As próximas células são opcionais. **Não execute a geração completa antes de verificar os exemplos reais de `avaliacao_v8_2.csv` e registrar S/N.** Os dados de saída usarão nomes da v8.2, sem substituir os arquivos antigos.

In [ ]:
# 6. OPCIONAL — geração completa, após validar a qualidade da amostra
CONFIRME_AMOSTRA_REVISADA = False  # Altere para True SOMENTE após revisar a amostra
if not CONFIRME_AMOSTRA_REVISADA:
    raise RuntimeError("Geração completa bloqueada. Revise avaliacao_v8_2.csv primeiro.")

qa_segs, chat_segs = geracao.planejar_divisao(segs, n_qa=50, n_chat=30)
inicio = time.time()
qa, estat_qa = geracao.gerar_qa(qa_segs, gerar, total=100, pasta_cache=CACHE)
chat, estat_chat = geracao.gerar_chat(chat_segs, gerar, total=30, pasta_cache=CACHE)
print(f"Tempo: {(time.time()-inicio)/60:.1f} min")
print("QA:", estat_qa)
print("CHAT:", estat_chat)

In [ ]:
# 7. OPCIONAL — exportar planilha de revisão (requer executar a célula 6)
if "qa" not in globals() or "chat" not in globals():
    raise RuntimeError("Execute a geração completa (célula 6) antes.")
amostra_qa = {e["id"] for e in geracao.amostra_revisao(qa)}
amostra_chat = {e["id"] for e in geracao.amostra_revisao(chat)}
SAIDA_FINAL = Path("/content/drive/MyDrive/radar-cidadao/revisao_geracao_v8_2.csv")
if SAIDA_FINAL.exists():
    raise FileExistsError(f"Planilha existente preservada: {SAIDA_FINAL}")
with SAIDA_FINAL.open("w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f, delimiter=";")
    w.writerow(["id","amostra 20%","segmento_id","fonte","paginas",
                "pergunta / diálogo","resposta","ok? (S/N)","observação"])
    for e in qa:
        w.writerow([e["id"],"sim" if e["id"] in amostra_qa else "",e["segmento_id"],
                    e["fonte"],e["paginas"],e["pergunta"],e["resposta"],"",""])
    for e in chat:
        dialogo = "\n".join(f"{m['role']}: {m['content']}" for m in e["mensagens"][1:])
        w.writerow([e["id"],"sim" if e["id"] in amostra_chat else "",e["segmento_id"],
                    e["fonte"],e["paginas"],dialogo,"","",""])
print("Planilha para revisão:",SAIDA_FINAL)

In [ ]:
# 8. OPCIONAL — exportar cópia do cache v8.2 (não remove os originais)
import zipfile
ARQUIVO_ZIP = Path("/content/cache_geracao_v8_2.zip")
arquivos = sorted(CACHE.glob("*_v8_2_*.json"))
with zipfile.ZipFile(ARQUIVO_ZIP,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for arq in arquivos:
        z.write(arq,arcname=arq.name)
print("Arquivos incluídos:",len(arquivos),"| ZIP:",ARQUIVO_ZIP)
from google.colab import files
files.download(str(ARQUIVO_ZIP))

In [19]:
# Preparar rotinas locais de exportação (não altera o código do projeto)
from pathlib import Path
Path('/content/fechar_entrega.py').write_text('"""Radar Cidadao: exportacao conservadora de dados existentes, verificacao e relatorio.\nNao aprova automaticamente fidelidade; nao cria exemplos inexistentes.\n"""\nfrom __future__ import annotations\nimport csv, hashlib, json, math, random, re, statistics, unicodedata\nfrom collections import Counter\nfrom datetime import datetime\nfrom pathlib import Path\n\nMINIMOS = {\'texto\':50,\'qa\':100,\'chat\':30}\n\ndef _campo(o, nome, default=None): return o.get(nome, default) if isinstance(o,dict) else getattr(o,nome,default)\ndef _norm(s):\n    t=unicodedata.normalize(\'NFKD\',str(s)).casefold()\n    return re.sub(r\'\\s+\',\' \',re.sub(r\'[^a-z0-9 ]\',\' \', \'\'.join(c for c in t if not unicodedata.combining(c)))).strip()\ndef _chave(e, tipo):\n    if tipo==\'texto\': s=e[\'text\']\n    elif tipo==\'qa\': s=e[\'instruction\']\n    else: s=\'\\n\'.join(m[\'role\']+\':\'+m[\'content\'] for m in e[\'messages\'] if m[\'role\']!=\'system\')\n    return hashlib.sha256(_norm(s).encode()).hexdigest()\ndef _conteudo(seg):\n    for nome in (\'texto\',\'text\',\'conteudo\'):\n        v=_campo(seg,nome)\n        if isinstance(v,str) and v.strip(): return v.strip()\n    return \'\'\ndef normalizar_texto(segs):\n    saida=[]\n    for seg in segs:\n        fonte=_campo(seg,\'fonte\'); txt=_conteudo(seg)\n        if fonte and txt:\n            saida.append({\'text\':txt,\'fonte\':str(fonte),\'segmento_id\':str(_campo(seg,\'id\',\'\'))})\n    return saida\n\ndef normalizar_qa(itens):\n    resultado=[]\n    for e in itens:\n        pergunta=_campo(e,\'pergunta\',\'\'); resposta=_campo(e,\'resposta\',\'\'); fonte=_campo(e,\'fonte\',\'\')\n        if pergunta and resposta and fonte:\n            resultado.append({\'instruction\':pergunta.strip(),\'input\':\'\',\'output\':resposta.strip(), \'fonte\':str(fonte), \'segmento_id\':str(_campo(e,\'segmento_id\',\'\'))})\n    return resultado\n\ndef normalizar_chat(itens):\n    resultado=[]\n    for e in itens:\n        ms=_campo(e,\'mensagens\',[]); fonte=_campo(e,\'fonte\')\n        if not fonte or not isinstance(ms,list): continue\n        mensagens=[{\'role\':_campo(m,\'role\',\'\'), \'content\':_campo(m,\'content\',\'\')} for m in ms]\n        resultado.append({\'messages\':mensagens,\'fonte\':str(fonte),\'tipo\':str(_campo(e,\'tipo\',\'chat\')), \'segmento_id\':str(_campo(e,\'segmento_id\',\'\') or \'\')})\n    return resultado\n\ndef conferir_formato(e,tipo):\n    if not isinstance(e,dict) or not isinstance(e.get(\'fonte\'),str) or not e[\'fonte\'].strip(): return \'fonte ausente\'\n    if tipo==\'texto\':\n        if not isinstance(e.get(\'text\'),str) or len(e[\'text\'].strip())<30:return \'texto curto/ausente\'\n    if tipo==\'qa\':\n        if not all(isinstance(e.get(k),str) for k in [\'instruction\',\'input\',\'output\']):return \'campos QA ausentes\'\n        if len(e[\'instruction\'].strip())<12 or len(e[\'output\'].strip())<12:return \'pergunta/resposta curta\'\n    if tipo==\'chat\':\n        ms=e.get(\'messages\')\n        if not isinstance(ms,list):return \'mensagens ausentes\'\n        papeis=[m.get(\'role\') for m in ms if isinstance(m,dict) and m.get(\'role\')!=\'system\']\n        if len(papeis)<4 or len(papeis)%2 or papeis != [\'user\',\'assistant\']*(len(papeis)//2): return \'menos de 2 trocas ou papeis invalidos\'\n        if any(not isinstance(m,dict) or not isinstance(m.get(\'content\'),str) or not m[\'content\'].strip() or m.get(\'role\') not in (\'system\',\'user\',\'assistant\') for m in ms):return \'mensagem vazia/papel invalido\'\n    return None\n\ndef filtrar(itens,tipo):\n    validos=[]; duplicatas=0; invalidos=[]; vistos=set()\n    for n,e in enumerate(itens,1):\n        problema=conferir_formato(e,tipo)\n        if problema:\n            invalidos.append({\'linha\':n,\'problema\':problema});continue\n        k=_chave(e,tipo)\n        if k in vistos: duplicatas+=1;continue\n        vistos.add(k);validos.append(e)\n    return validos,duplicatas,invalidos\n\ndef salvar_jsonl(p,itens):\n    with Path(p).open(\'w\',encoding=\'utf-8\') as f:\n        for e in itens:f.write(json.dumps(e,ensure_ascii=False)+\'\\n\')\n\ndef ler_jsonl(p):\n    itens=[]\n    with Path(p).open(encoding=\'utf-8\') as f:\n        for n,l in enumerate(f,1):\n            if l.strip():\n                try:itens.append(json.loads(l))\n                except json.JSONDecodeError as ex:raise ValueError(f\'{p}: linha {n}: {ex}\') from ex\n    return itens\n\ndef estimar_tokens(e):return len(re.findall(r\'\\w+|[^\\w\\s]\',json.dumps(e,ensure_ascii=False)))\n\ndef exportar(destino,segs,qa,chat,recusas=(),fonte_documentos=None):\n    destino=Path(destino);destino.mkdir(parents=True,exist_ok=True)\n    brutos={\'texto\':normalizar_texto(segs),\'qa\':normalizar_qa(qa),\'chat\':normalizar_chat(list(chat)+list(recusas))}\n    metr={};amostras={}\n    for tipo,original in brutos.items():\n        validos,dup,invalidos=filtrar(original,tipo)\n        salvar_jsonl(destino/f\'corpus_{tipo}.jsonl\',validos)\n        metr[tipo]={\'quantidade\':len(validos),\'minimo\':MINIMOS[tipo],\'quantidade_atingida\':len(validos)>=MINIMOS[tipo], \'media_tokens_aprox\':round(statistics.mean(map(estimar_tokens,validos)),1) if validos else 0,\'duplicatas_removidas\':dup,\'invalidos_removidos\':len(invalidos),\'detalhes_invalidos\':invalidos[:40]}\n        amostras[tipo]=random.Random(2026).sample(validos,min(5,len(validos)))\n    metr[\'recusas_validas\']=sum(1 for e in ler_jsonl(destino/\'corpus_chat.jsonl\') if e.get(\'tipo\')==\'recusa\')\n    metr[\'recusas_atingidas\']=metr[\'recusas_validas\']>=10\n    metr[\'revisao_humana_confirmada\']=False\n    metr[\'fontes_documentais_conferidas\']=False\n    metr[\'observacao\']=\'Contagem e estrutura verificadas por codigo; fidelidade a fontes e revisao de >=20% exigem verificacao humana.\'\n    metr[\'pronto_para_entrega\']=all(metr[k][\'quantidade_atingida\'] for k in MINIMOS) and metr[\'recusas_atingidas\'] and metr[\'revisao_humana_confirmada\'] and metr[\'fontes_documentais_conferidas\']\n    (destino/\'estatisticas.json\').write_text(json.dumps(metr,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    (destino/\'amostras_5_por_corpus.json\').write_text(json.dumps(amostras,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    criar_revisao(destino)\n    return metr\n\ndef criar_revisao(destino):\n    destino=Path(destino); p=destino/\'revisao_humana_20_porcento.csv\'\n    # Criar sem sobrescrever uma revisao ja preenchida\n    if p.exists():return p\n    qa=ler_jsonl(destino/\'corpus_qa.jsonl\'); chats=ler_jsonl(destino/\'corpus_chat.jsonl\')\n    rng=random.Random(2026); selecionados=[]\n    for tipo,lista in ((\'qa\',qa),(\'chat\',chats)):\n        n=math.ceil(.2*len(lista));selecionados.extend((tipo,i,e) for i,e in enumerate(rng.sample(lista,n),1))\n    with p.open(\'w\',encoding=\'utf-8-sig\',newline=\'\') as f:\n        w=csv.writer(f,delimiter=\';\');w.writerow([\'tipo\',\'numero_amostra\',\'fonte\',\'pergunta_ou_dialogo\',\'resposta\',\'apoio_na_fonte_S_N\',\'responde_S_N\',\'seguranca_S_N\',\'revisor\',\'observacoes\'])\n        for tipo,i,e in selecionados:\n            texto=e[\'instruction\'] if tipo==\'qa\' else \'\\n\'.join(m[\'role\']+\': \'+m[\'content\'] for m in e[\'messages\'] if m[\'role\']!=\'system\')\n            resposta=e[\'output\'] if tipo==\'qa\' else \'\'\n            w.writerow([tipo,i,e[\'fonte\'],texto,resposta,\'\',\'\',\'\',\'\',\'\'])\n    return p\n\ndef auditar_pasta(pasta):\n    pasta=Path(pasta); ret={}\n    for tipo in MINIMOS:\n        p=pasta/f\'corpus_{tipo}.jsonl\'\n        if not p.exists(): ret[tipo]={\'erro\':\'arquivo nao existe\'};continue\n        data=ler_jsonl(p);validos,dup,invalidos=filtrar(data,tipo)\n        ret[tipo]={\'quantidade\':len(data),\'quantidade_valida\':len(validos),\'minimo\':MINIMOS[tipo],\'duplicatas\':dup,\'invalidos\':invalidos,\'ok\':len(data)==len(validos) and len(validos)>=MINIMOS[tipo]}\n    if (pasta/\'corpus_chat.jsonl\').exists():\n        ret[\'recusas\']=sum(1 for e in ler_jsonl(pasta/\'corpus_chat.jsonl\') if e.get(\'tipo\')==\'recusa\')\n    return ret\n\nif __name__==\'__main__\':\n    import argparse\n    a=argparse.ArgumentParser();a.add_argument(\'pasta\');args=a.parse_args()\n    print(json.dumps(auditar_pasta(args.pasta),ensure_ascii=False,indent=2))\n', encoding='utf-8')
Path('/content/relatorio_pdf.py').write_text('"""Gera relatorio tecnico de no maximo 6 paginas, sem inventar aprovacao humana."""\nimport json\nfrom pathlib import Path\nfrom reportlab.lib import colors\nfrom reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle\nfrom reportlab.lib.enums import TA_CENTER\nfrom reportlab.lib.pagesizes import A4\nfrom reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, KeepTogether\n\n\ndef construir(pasta):\n    pasta=Path(pasta);met=json.loads((pasta/\'estatisticas.json\').read_text(encoding=\'utf-8\'))\n    p=pasta/\'relatorio_GS_Radar_Cidadao_RASCUNHO.pdf\'\n    st=getSampleStyleSheet();st.add(ParagraphStyle(name=\'CorpoGS\',parent=st[\'BodyText\'],fontSize=10,leading=15,spaceAfter=10))\n    st.add(ParagraphStyle(name=\'TitGS\',parent=st[\'Title\'],textColor=colors.HexColor(\'#183B50\'),spaceAfter=15))\n    st.add(ParagraphStyle(name=\'SubGS\',parent=st[\'Heading2\'],textColor=colors.HexColor(\'#183B50\'),spaceBefore=14,spaceAfter=7))\n    doc=SimpleDocTemplate(str(p),pagesize=A4,leftMargin=48,rightMargin=48,topMargin=47,bottomMargin=47)\n    h=lambda s:Paragraph(s,st[\'SubGS\']);t=lambda s:Paragraph(s,st[\'CorpoGS\'])\n    blocos=[Paragraph(\'Radar Cidadão — Fábrica de Corpus\',st[\'TitGS\']), t(\'<b>Relatório técnico — rascunho sujeito à revisão da equipe.</b> Não declarar este PDF final enquanto houver campos pendentes.\'),\n        h(\'1. Problema, público e ODS\'),t(\'Tema: prevenção de riscos associados a enchentes, chuvas intensas e deslizamentos. Persona: morador de área suscetível a desastres, que precisa de orientação pública em linguagem acessível. ODS relacionado: 11 (Cidades e Comunidades Sustentáveis).\'),\n        h(\'2. Documentos e ingestão\'),t(\'Fontes: documentos de órgãos públicos da área de saúde, cidades e proteção civil. <b>PENDENTE:</b> listar os arquivos reais utilizados, URLs, datas e número total de páginas. A ferramenta deve demonstrar PDF, TXT, DOCX, HTML e texto colado.\'),\n        h(\'3. Processamento\'),t(\'Esteira Python: extração de conteúdo com rastreabilidade; limpeza de ruído; normalização em UTF-8; anonimização anterior à geração; segmentação preservando frases e parágrafos. <b>PENDENTE:</b> anexar um exemplo comprovado de antes/depois da anonimização e medir o tamanho dos segmentos entre 256 e 512 tokens.\'),\n        h(\'4. Geração e estatísticas\')]\n    data=[[\'Corpus\',\'Válidos\',\'Mínimo\',\'Média aprox. tokens\',\'Duplicatas\']]\n    for k in (\'texto\',\'qa\',\'chat\'):\n        v=met[k];data.append([k,str(v[\'quantidade\']),str(v[\'minimo\']),str(v[\'media_tokens_aprox\']),str(v[\'duplicatas_removidas\'])])\n    tb=Table(data,colWidths=[90,65,65,130,85],repeatRows=1,hAlign=\'LEFT\');tb.setStyle(TableStyle([\n        (\'BACKGROUND\',(0,0),(-1,0),colors.HexColor(\'#183B50\')),(\'TEXTCOLOR\',(0,0),(-1,0),colors.white),(\'ROWBACKGROUNDS\',(0,1),(-1,-1),[colors.white,colors.HexColor(\'#F0F5F7\')]),(\'VALIGN\',(0,0),(-1,-1),\'MIDDLE\'),(\'TOPPADDING\',(0,0),(-1,-1),9),(\'BOTTOMPADDING\',(0,0),(-1,-1),9)]))\n    blocos += [tb,Spacer(1,12),t(f"Recusas estruturais: <b>{met[\'recusas_validas\']}</b> (mínimo: 10). As médias de tokens são <b>estimativas lexicais</b>, não medições com o tokenizador do modelo. Cinco amostras aleatórias de cada corpus foram salvas em <b>amostras_5_por_corpus.json</b>."),\n        h(\'5. Qualidade, riscos e LGPD\'),t(\'O projeto utiliza Qwen2.5-14B-Instruct para auxiliar a geração, com prompt solicitando apoio estrito no texto-fonte, clareza e respeito a quem executa a ação. Filtros automáticos removem entradas inválidas e duplicatas exatas. Os filtros NÃO comprovam fidelidade factual. <b>PENDENTE:</b> equipe revisar pelo menos 20% dos exemplos de IA em revisao_humana_20_porcento.csv e conferir as fontes; corrigir ou excluir exemplos inadequados antes da entrega.\'),\n        h(\'6. Limitações e dificuldades\'),t(\'Desafios observados: respostas repetidas, troca entre responsabilidade do morador e de órgãos públicos, respostas sobre saúde incompletas e divergência entre juiz automático e revisão humana. Exemplos de recusa foram acrescentados para pedidos fora de escopo. Nenhuma taxa de qualidade humana deve ser declarada antes da revisão.\'),\n        h(\'7. Reprodutibilidade e demonstração\'),t(\'Código e arquivos deverão ser disponibilizados em um único link público/compartilhado. <b>PENDENTE:</b> inserir nomes completos da equipe, confirmar execução de ponta a ponta e gravar vídeo de até 5 minutos mostrando upload, processamento, geração dos três JSONL, estatísticas e exemplos.\')]\n    doc.build(blocos)\n    return p\nif __name__==\'__main__\':\n    import sys\n    print(construir(sys.argv[1]))\n', encoding='utf-8')
Path('/content/README_ENTREGA.md').write_text('# Radar Cidadão — GS 2027: Fábrica de Corpus CitizenTech\n\nEste pacote **não contém corpus inventado**. Ele fecha o processo com dados realmente gerados pelo projeto e valida exigências formais do enunciado.\n\n## Como usar no Colab, sem refazer as gerações já executadas\n\n1. No notebook em que estão carregados `segs`, `gerar`, `CACHE` e `geracao`, execute `FECHAMENTO_GS_19_NOV.ipynb` no mesmo runtime (ou copie as células para o notebook aberto). Se a sessão tiver reiniciado, execute as células de preparação do notebook original antes de prosseguir.\n2. A primeira célula usa o código local do GitHub, commit 2907161, e não modifica o repositório.\n3. A geração final só é executada se `qa` e `chat` existentes não tiverem as quantidades necessárias. Ela usa o mesmo cache (`CACHE`) e os mesmos segmentos, evitando retrabalho de consultas anteriores. A **geração pode ser lenta** se o cache ainda não contiver os resultados da escala completa.\n4. Os JSONL, estatísticas, cinco amostras por corpus, planilha para revisão humana e relatório preliminar ficam em uma **nova pasta** do Drive `radar-cidadao/entrega_GS_19nov`, sem sobrescrever as v8.2/v8.3.\n5. Preencha a planilha de revisão humana, confronte cada exemplo com o documento citado e inclua uma evidência antes/depois de anonimização no PDF.\n6. Acrescente ao README final nomes completos dos integrantes, fontes exatas (documentos/URLs), bibliotecas, instruções para rodar e vídeo de até cinco minutos. Não declare o pacote final antes de passar em todos os critérios.\n\n## Quantidades e formatos\n\n- `corpus_texto.jsonl`: mínimo de 50 trechos, `text`, `fonte`.\n- `corpus_qa.jsonl`: mínimo de 100 pares, `instruction`, `input`, `output`, `fonte`.\n- `corpus_chat.jsonl`: mínimo de 30 diálogos, `messages` com `role`/`content`, pelo menos duas trocas e 10 recusas.\n- `estatisticas.json`: quantidades, duplicatas, estimativas de tokens, erros. **Não equivale a validação semântica.**\n- `revisao_humana_20_porcento.csv`: marcar S/N após conferir a fonte. A revisão não é preenchida automaticamente.\n\n## Aviso de qualidade\n\nO módulo exporta o que o modelo gerou, mas não inventa respostas para alcançar a meta. A validação automática detecta problemas estruturais e repetição da mesma pergunta; **não garante 90% de qualidade nem comprova que a resposta esteja no PDF**.\n\n## Bibliotecas\n\nPython, pacote original `nlp-fabrica-corpus` (spaCy, bibliotecas de extração, transformers, PyTorch), `reportlab` para o relatório PDF, bibliotecas-padrão `csv`, `json`, `hashlib`, `pathlib`.\n\n## Integrantes\n\n**PENDENTE: preencher nomes completos de todos os integrantes antes da entrega.**\n', encoding='utf-8')
print('Rotinas locais de fechamento disponíveis; não houve modificação no GitHub.')

Rotinas locais de fechamento disponíveis; não houve modificação no GitHub.


In [20]:
# 1. Conferir runtime, corpus e versão SEM reiniciar a GPU
import sys, subprocess, importlib
from pathlib import Path
PROJETO=Path('/content/radar-cidadao/nlp-fabrica-corpus')
if str(PROJETO/'src') not in sys.path: sys.path.insert(0,str(PROJETO/'src'))
import fabrica_corpus.geracao as geracao
from fabrica_corpus import geracao as geracao
print('Versão QA/Chat:',getattr(geracao,'VERSAO_QA_CHAT','não encontrada'))
print('Versão recusas:',geracao.VERSAO_PROMPT)
print('Segmentos disponíveis:',len(globals().get('segs',[])))
print('Gerador Qwen na sessão:', callable(globals().get('gerar',None)))
assert getattr(geracao,'VERSAO_QA_CHAT',None)=='v8_3','A versão do Colab deve ser v8_3. Atualize o código e recarregue o módulo.'
assert 'segs' in globals() and len(segs)>=50, 'Execute a etapa de ingestão/segmentação do seu notebook primeiro.'
assert 'CACHE' in globals() and Path(CACHE).exists(), 'Reutilize o cache existente no Drive.'
assert Path('/content/fechar_entrega.py').exists(), 'Faça upload dos arquivos fechar_entrega.py e relatorio_pdf.py para /content.'
import fechar_entrega as fechar

Versão QA/Chat: v8_3
Versão recusas: v8_2
Segmentos disponíveis: 306
Gerador Qwen na sessão: True


In [21]:
# 2. Preparar gerações completas, aproveitando dados existentes e cache
# As amostras anteriores (14, 19, 4...) NÃO bastam para a entrega.
# Se uma sessão anterior já gerou o corpus completo, preserva os objetos `qa` e `chat`.
from pathlib import Path
import time
qa_antes=len(globals().get('qa',[]))
chat_antes=len(globals().get('chat',[]))
print('QA já na memória:',qa_antes,'| chat já na memória:',chat_antes)
precisa_qa=qa_antes<100
precisa_chat=chat_antes<20
if precisa_qa or precisa_chat:
    assert callable(globals().get('gerar',None)), 'Para gerar os itens faltantes, carregue primeiro o Qwen no notebook original.'
if precisa_qa:
    print('Gerando até 100 QA (cache da v8.3 preservado)...')
    qa, estat_qa=geracao.gerar_qa(segs,gerar,total=100,por_segmento=2,pasta_cache=CACHE,filtrar_repeticoes=True)
    print('QA geradas:',len(qa),'|',estat_qa)
if precisa_chat:
    print('Gerando até 20 diálogos comuns (10 recusas completarão o mínimo de 30)...')
    chat, estat_chat=geracao.gerar_chat(segs,gerar,total=20,pasta_cache=CACHE)
    print('Diálogos comuns:',len(chat),'|',estat_chat)
# Recusas já cacheadas: não reexecutar 11 chamadas ao Qwen se o cache tiver os resultados
if 'recusas' not in globals() or len(recusas)<10:
    assert callable(globals().get('gerar',None)), 'Carregue o Qwen para gerar recusas que ainda faltem.'
    recusas, estat_recusas=geracao.gerar_recusas(gerar,total=12,pasta_cache=CACHE)
print('Brutos | QA:',len(qa),'| Chat:',len(chat),'| Recusas:',len(recusas))

QA já na memória: 14 | chat já na memória: 4
Gerando até 100 QA (cache da v8.3 preservado)...
[perguntas] 2/100 | segmento mcid_2024_periferia_sem_risco_0003 | 2 aprovadas, 1 tentativa(s)
[perguntas] 2/100 | segmento mcid_2026_mapeamento_riscos_0071 | 0 aprovadas, 3 tentativa(s)
[perguntas] 4/100 | segmento ms_2017_cartilha_alerta_chuvas_0011 | 2 aprovadas, 2 tentativa(s)
[perguntas] 6/100 | segmento ms_2025_plano_contingencia_chuvas_0027 | 2 aprovadas, 3 tentativa(s)
[perguntas] 8/100 | segmento ms_2026_enchentes_0001 | 2 aprovadas, 1 tentativa(s)
[perguntas] 9/100 | segmento mcid_2024_periferia_sem_risco_0002 | 1 aprovadas, 3 tentativa(s)
[perguntas] 10/100 | segmento mcid_2026_mapeamento_riscos_0074 | 1 aprovadas, 3 tentativa(s)
[perguntas] 12/100 | segmento ms_2017_cartilha_alerta_chuvas_0001 | 2 aprovadas, 2 tentativa(s)
[perguntas] 14/100 | segmento ms_2025_plano_contingencia_chuvas_0039 | 2 aprovadas, 2 tentativa(s)
[perguntas] 15/100 | segmento ms_2026_enchentes_0002 | 1 aprova

In [22]:
# 3. Exportar/validar em nova pasta do Drive (não altera os anteriores)
from pathlib import Path
DESTINO=Path('/content/drive/MyDrive/radar-cidadao/entrega_GS_19nov')
if (DESTINO/'revisao_humana_20_porcento.csv').exists(): print('Revisão humana anterior será preservada, sem sobrescrita.')
metricas=fechar.exportar(DESTINO,segs,qa,chat,recusas)
for tipo in ('texto','qa','chat'):
    m=metricas[tipo]
    print(tipo,':',m['quantidade'],'/',m['minimo'],'| duplicatas:',m['duplicatas_removidas'],'| inválidos:',m['invalidos_removidos'])
print('Recusas:',metricas['recusas_validas'],'/10')
print('Estrutura/quantidades:',fechar.auditar_pasta(DESTINO))
if any(not metricas[k]['quantidade_atingida'] for k in ('texto','qa','chat')) or not metricas['recusas_atingidas']:
    print('ATENÇÃO: AINDA FALTAM QUANTIDADES OBRIGATÓRIAS. Não entregue este pacote incompleto.')

from pprint import pprint
import json
with (DESTINO/'amostras_5_por_corpus.json').open(encoding='utf-8') as f: amostras=json.load(f)
for nome, cinco in amostras.items():
    print('\nCinco exemplos aleatórios —',nome)
    for x in cinco:pprint(x,width=95)


texto : 306 / 50 | duplicatas: 0 | inválidos: 0
qa : 95 / 100 | duplicatas: 4 | inválidos: 1
chat : 31 / 30 | duplicatas: 0 | inválidos: 0
Recusas: 11 /10
Estrutura/quantidades: {'texto': {'quantidade': 306, 'quantidade_valida': 306, 'minimo': 50, 'duplicatas': 0, 'invalidos': [], 'ok': True}, 'qa': {'quantidade': 95, 'quantidade_valida': 95, 'minimo': 100, 'duplicatas': 0, 'invalidos': [], 'ok': False}, 'chat': {'quantidade': 31, 'quantidade_valida': 31, 'minimo': 30, 'duplicatas': 0, 'invalidos': [], 'ok': True}, 'recusas': 11}
ATENÇÃO: AINDA FALTAM QUANTIDADES OBRIGATÓRIAS. Não entregue este pacote incompleto.

Cinco exemplos aleatórios — texto
{'fonte': 'mcid_2026_mapeamento_riscos.pdf',
 'segmento_id': 'mcid_2026_mapeamento_riscos_0057',
 'text': 'Para tanto, propõe-se que a classificação do perigo se fundamente em dois pilares: '
         "a altura da lâmina d'água e a frequência de ocorrência em um intervalo de dez anos. "
         'A aplicação técnica desses critérios deve obse

In [23]:
# 4. Preparar relatório e ZIP com os dados efetivamente exportados
import subprocess,sys,zipfile,shutil
subprocess.run([sys.executable,'-m','pip','-q','install','reportlab'],check=True)
from relatorio_pdf import construir
relatorio=construir(DESTINO)
shutil.copy('/content/fechar_entrega.py',DESTINO/'fechar_entrega.py')
if Path('/content/README_ENTREGA.md').exists():shutil.copy('/content/README_ENTREGA.md',DESTINO/'README_ENTREGA.md')
zip_path=DESTINO.parent/'RADAR_CIDADAO_ENTREGA_RASCUNHO.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for f in DESTINO.rglob('*'):
        if f.is_file():z.write(f,arcname=f.relative_to(DESTINO.parent))
print('RELATÓRIO:',relatorio)
print('PACOTE:',zip_path)
print('REVISÃO HUMANA PENDENTE:', DESTINO/'revisao_humana_20_porcento.csv')
print('NÃO ENTREGAR sem confirmar fontes, revisão de >=20%, nomes, README e vídeo.')


RELATÓRIO: /content/drive/MyDrive/radar-cidadao/entrega_GS_19nov/relatorio_GS_Radar_Cidadao_RASCUNHO.pdf
PACOTE: /content/drive/MyDrive/radar-cidadao/RADAR_CIDADAO_ENTREGA_RASCUNHO.zip
REVISÃO HUMANA PENDENTE: /content/drive/MyDrive/radar-cidadao/entrega_GS_19nov/revisao_humana_20_porcento.csv
NÃO ENTREGAR sem confirmar fontes, revisão de >=20%, nomes, README e vídeo.


In [24]:
# 1. Montar Drive, ler a entrega anterior e preparar verificadores locais
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, sys, shutil, subprocess, importlib, math, random
BASE=Path('/content/drive/MyDrive/radar-cidadao')
ANTERIOR=BASE/'entrega_GS_19nov'
NOVO=BASE/'entrega_GS_19nov_corrigida'
REPO=Path('/content/radar-cidadao')
PROJETO=REPO/'nlp-fabrica-corpus'
for nome in ('corpus_texto.jsonl','corpus_qa.jsonl','corpus_chat.jsonl'):
    assert (ANTERIOR/nome).is_file(), f'Falta arquivo anterior no Drive: {ANTERIOR/nome}'
assert (PROJETO/'src'/'fabrica_corpus'/'geracao.py').exists(), ('Repositório não encontrado no Colab. Execute a célula de configuração do notebook original (sem executar a geração), ou clone o repositório.')
src=str(PROJETO/'src')
if src not in sys.path: sys.path.insert(0,src)
sys.path.insert(0,'/content')
# Helpers de exportação correspondentes à última entrega. Não modificam o GitHub.
Path('/content/fechar_entrega.py').write_text('"""Radar Cidadao: exportacao conservadora de dados existentes, verificacao e relatorio.\nNao aprova automaticamente fidelidade; nao cria exemplos inexistentes.\n"""\nfrom __future__ import annotations\nimport csv, hashlib, json, math, random, re, statistics, unicodedata\nfrom collections import Counter\nfrom datetime import datetime\nfrom pathlib import Path\n\nMINIMOS = {\'texto\':50,\'qa\':100,\'chat\':30}\n\ndef _campo(o, nome, default=None): return o.get(nome, default) if isinstance(o,dict) else getattr(o,nome,default)\ndef _norm(s):\n    t=unicodedata.normalize(\'NFKD\',str(s)).casefold()\n    return re.sub(r\'\\s+\',\' \',re.sub(r\'[^a-z0-9 ]\',\' \', \'\'.join(c for c in t if not unicodedata.combining(c)))).strip()\ndef _chave(e, tipo):\n    if tipo==\'texto\': s=e[\'text\']\n    elif tipo==\'qa\': s=e[\'instruction\']\n    else: s=\'\\n\'.join(m[\'role\']+\':\'+m[\'content\'] for m in e[\'messages\'] if m[\'role\']!=\'system\')\n    return hashlib.sha256(_norm(s).encode()).hexdigest()\ndef _conteudo(seg):\n    for nome in (\'texto\',\'text\',\'conteudo\'):\n        v=_campo(seg,nome)\n        if isinstance(v,str) and v.strip(): return v.strip()\n    return \'\'\ndef normalizar_texto(segs):\n    saida=[]\n    for seg in segs:\n        fonte=_campo(seg,\'fonte\'); txt=_conteudo(seg)\n        if fonte and txt:\n            saida.append({\'text\':txt,\'fonte\':str(fonte),\'segmento_id\':str(_campo(seg,\'id\',\'\'))})\n    return saida\n\ndef normalizar_qa(itens):\n    resultado=[]\n    for e in itens:\n        pergunta=_campo(e,\'pergunta\',\'\'); resposta=_campo(e,\'resposta\',\'\'); fonte=_campo(e,\'fonte\',\'\')\n        if pergunta and resposta and fonte:\n            resultado.append({\'instruction\':pergunta.strip(),\'input\':\'\',\'output\':resposta.strip(), \'fonte\':str(fonte), \'segmento_id\':str(_campo(e,\'segmento_id\',\'\'))})\n    return resultado\n\ndef normalizar_chat(itens):\n    resultado=[]\n    for e in itens:\n        ms=_campo(e,\'mensagens\',[]); fonte=_campo(e,\'fonte\')\n        if not fonte or not isinstance(ms,list): continue\n        mensagens=[{\'role\':_campo(m,\'role\',\'\'), \'content\':_campo(m,\'content\',\'\')} for m in ms]\n        resultado.append({\'messages\':mensagens,\'fonte\':str(fonte),\'tipo\':str(_campo(e,\'tipo\',\'chat\')), \'segmento_id\':str(_campo(e,\'segmento_id\',\'\') or \'\')})\n    return resultado\n\ndef conferir_formato(e,tipo):\n    if not isinstance(e,dict) or not isinstance(e.get(\'fonte\'),str) or not e[\'fonte\'].strip(): return \'fonte ausente\'\n    if tipo==\'texto\':\n        if not isinstance(e.get(\'text\'),str) or len(e[\'text\'].strip())<30:return \'texto curto/ausente\'\n    if tipo==\'qa\':\n        if not all(isinstance(e.get(k),str) for k in [\'instruction\',\'input\',\'output\']):return \'campos QA ausentes\'\n        if len(e[\'instruction\'].strip())<12 or len(e[\'output\'].strip())<12:return \'pergunta/resposta curta\'\n    if tipo==\'chat\':\n        ms=e.get(\'messages\')\n        if not isinstance(ms,list):return \'mensagens ausentes\'\n        papeis=[m.get(\'role\') for m in ms if isinstance(m,dict) and m.get(\'role\')!=\'system\']\n        if len(papeis)<4 or len(papeis)%2 or papeis != [\'user\',\'assistant\']*(len(papeis)//2): return \'menos de 2 trocas ou papeis invalidos\'\n        if any(not isinstance(m,dict) or not isinstance(m.get(\'content\'),str) or not m[\'content\'].strip() or m.get(\'role\') not in (\'system\',\'user\',\'assistant\') for m in ms):return \'mensagem vazia/papel invalido\'\n    return None\n\ndef filtrar(itens,tipo):\n    validos=[]; duplicatas=0; invalidos=[]; vistos=set()\n    for n,e in enumerate(itens,1):\n        problema=conferir_formato(e,tipo)\n        if problema:\n            invalidos.append({\'linha\':n,\'problema\':problema});continue\n        k=_chave(e,tipo)\n        if k in vistos: duplicatas+=1;continue\n        vistos.add(k);validos.append(e)\n    return validos,duplicatas,invalidos\n\ndef salvar_jsonl(p,itens):\n    with Path(p).open(\'w\',encoding=\'utf-8\') as f:\n        for e in itens:f.write(json.dumps(e,ensure_ascii=False)+\'\\n\')\n\ndef ler_jsonl(p):\n    itens=[]\n    with Path(p).open(encoding=\'utf-8\') as f:\n        for n,l in enumerate(f,1):\n            if l.strip():\n                try:itens.append(json.loads(l))\n                except json.JSONDecodeError as ex:raise ValueError(f\'{p}: linha {n}: {ex}\') from ex\n    return itens\n\ndef estimar_tokens(e):return len(re.findall(r\'\\w+|[^\\w\\s]\',json.dumps(e,ensure_ascii=False)))\n\ndef exportar(destino,segs,qa,chat,recusas=(),fonte_documentos=None):\n    destino=Path(destino);destino.mkdir(parents=True,exist_ok=True)\n    brutos={\'texto\':normalizar_texto(segs),\'qa\':normalizar_qa(qa),\'chat\':normalizar_chat(list(chat)+list(recusas))}\n    metr={};amostras={}\n    for tipo,original in brutos.items():\n        validos,dup,invalidos=filtrar(original,tipo)\n        salvar_jsonl(destino/f\'corpus_{tipo}.jsonl\',validos)\n        metr[tipo]={\'quantidade\':len(validos),\'minimo\':MINIMOS[tipo],\'quantidade_atingida\':len(validos)>=MINIMOS[tipo], \'media_tokens_aprox\':round(statistics.mean(map(estimar_tokens,validos)),1) if validos else 0,\'duplicatas_removidas\':dup,\'invalidos_removidos\':len(invalidos),\'detalhes_invalidos\':invalidos[:40]}\n        amostras[tipo]=random.Random(2026).sample(validos,min(5,len(validos)))\n    metr[\'recusas_validas\']=sum(1 for e in ler_jsonl(destino/\'corpus_chat.jsonl\') if e.get(\'tipo\')==\'recusa\')\n    metr[\'recusas_atingidas\']=metr[\'recusas_validas\']>=10\n    metr[\'revisao_humana_confirmada\']=False\n    metr[\'fontes_documentais_conferidas\']=False\n    metr[\'observacao\']=\'Contagem e estrutura verificadas por codigo; fidelidade a fontes e revisao de >=20% exigem verificacao humana.\'\n    metr[\'pronto_para_entrega\']=all(metr[k][\'quantidade_atingida\'] for k in MINIMOS) and metr[\'recusas_atingidas\'] and metr[\'revisao_humana_confirmada\'] and metr[\'fontes_documentais_conferidas\']\n    (destino/\'estatisticas.json\').write_text(json.dumps(metr,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    (destino/\'amostras_5_por_corpus.json\').write_text(json.dumps(amostras,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    criar_revisao(destino)\n    return metr\n\ndef criar_revisao(destino):\n    destino=Path(destino); p=destino/\'revisao_humana_20_porcento.csv\'\n    # Criar sem sobrescrever uma revisao ja preenchida\n    if p.exists():return p\n    qa=ler_jsonl(destino/\'corpus_qa.jsonl\'); chats=ler_jsonl(destino/\'corpus_chat.jsonl\')\n    rng=random.Random(2026); selecionados=[]\n    for tipo,lista in ((\'qa\',qa),(\'chat\',chats)):\n        n=math.ceil(.2*len(lista));selecionados.extend((tipo,i,e) for i,e in enumerate(rng.sample(lista,n),1))\n    with p.open(\'w\',encoding=\'utf-8-sig\',newline=\'\') as f:\n        w=csv.writer(f,delimiter=\';\');w.writerow([\'tipo\',\'numero_amostra\',\'fonte\',\'pergunta_ou_dialogo\',\'resposta\',\'apoio_na_fonte_S_N\',\'responde_S_N\',\'seguranca_S_N\',\'revisor\',\'observacoes\'])\n        for tipo,i,e in selecionados:\n            texto=e[\'instruction\'] if tipo==\'qa\' else \'\\n\'.join(m[\'role\']+\': \'+m[\'content\'] for m in e[\'messages\'] if m[\'role\']!=\'system\')\n            resposta=e[\'output\'] if tipo==\'qa\' else \'\'\n            w.writerow([tipo,i,e[\'fonte\'],texto,resposta,\'\',\'\',\'\',\'\',\'\'])\n    return p\n\ndef auditar_pasta(pasta):\n    pasta=Path(pasta); ret={}\n    for tipo in MINIMOS:\n        p=pasta/f\'corpus_{tipo}.jsonl\'\n        if not p.exists(): ret[tipo]={\'erro\':\'arquivo nao existe\'};continue\n        data=ler_jsonl(p);validos,dup,invalidos=filtrar(data,tipo)\n        ret[tipo]={\'quantidade\':len(data),\'quantidade_valida\':len(validos),\'minimo\':MINIMOS[tipo],\'duplicatas\':dup,\'invalidos\':invalidos,\'ok\':len(data)==len(validos) and len(validos)>=MINIMOS[tipo]}\n    if (pasta/\'corpus_chat.jsonl\').exists():\n        ret[\'recusas\']=sum(1 for e in ler_jsonl(pasta/\'corpus_chat.jsonl\') if e.get(\'tipo\')==\'recusa\')\n    return ret\n\nif __name__==\'__main__\':\n    import argparse\n    a=argparse.ArgumentParser();a.add_argument(\'pasta\');args=a.parse_args()\n    print(json.dumps(auditar_pasta(args.pasta),ensure_ascii=False,indent=2))\n',encoding='utf-8')
Path('/content/relatorio_pdf.py').write_text('"""Gera relatorio tecnico de no maximo 6 paginas, sem inventar aprovacao humana."""\nimport json\nfrom pathlib import Path\nfrom reportlab.lib import colors\nfrom reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle\nfrom reportlab.lib.enums import TA_CENTER\nfrom reportlab.lib.pagesizes import A4\nfrom reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, KeepTogether\n\n\ndef construir(pasta):\n    pasta=Path(pasta);met=json.loads((pasta/\'estatisticas.json\').read_text(encoding=\'utf-8\'))\n    p=pasta/\'relatorio_GS_Radar_Cidadao_RASCUNHO.pdf\'\n    st=getSampleStyleSheet();st.add(ParagraphStyle(name=\'CorpoGS\',parent=st[\'BodyText\'],fontSize=10,leading=15,spaceAfter=10))\n    st.add(ParagraphStyle(name=\'TitGS\',parent=st[\'Title\'],textColor=colors.HexColor(\'#183B50\'),spaceAfter=15))\n    st.add(ParagraphStyle(name=\'SubGS\',parent=st[\'Heading2\'],textColor=colors.HexColor(\'#183B50\'),spaceBefore=14,spaceAfter=7))\n    doc=SimpleDocTemplate(str(p),pagesize=A4,leftMargin=48,rightMargin=48,topMargin=47,bottomMargin=47)\n    h=lambda s:Paragraph(s,st[\'SubGS\']);t=lambda s:Paragraph(s,st[\'CorpoGS\'])\n    blocos=[Paragraph(\'Radar Cidadão — Fábrica de Corpus\',st[\'TitGS\']), t(\'<b>Relatório técnico — rascunho sujeito à revisão da equipe.</b> Não declarar este PDF final enquanto houver campos pendentes.\'),\n        h(\'1. Problema, público e ODS\'),t(\'Tema: prevenção de riscos associados a enchentes, chuvas intensas e deslizamentos. Persona: morador de área suscetível a desastres, que precisa de orientação pública em linguagem acessível. ODS relacionado: 11 (Cidades e Comunidades Sustentáveis).\'),\n        h(\'2. Documentos e ingestão\'),t(\'Fontes: documentos de órgãos públicos da área de saúde, cidades e proteção civil. <b>PENDENTE:</b> listar os arquivos reais utilizados, URLs, datas e número total de páginas. A ferramenta deve demonstrar PDF, TXT, DOCX, HTML e texto colado.\'),\n        h(\'3. Processamento\'),t(\'Esteira Python: extração de conteúdo com rastreabilidade; limpeza de ruído; normalização em UTF-8; anonimização anterior à geração; segmentação preservando frases e parágrafos. <b>PENDENTE:</b> anexar um exemplo comprovado de antes/depois da anonimização e medir o tamanho dos segmentos entre 256 e 512 tokens.\'),\n        h(\'4. Geração e estatísticas\')]\n    data=[[\'Corpus\',\'Válidos\',\'Mínimo\',\'Média aprox. tokens\',\'Duplicatas\']]\n    for k in (\'texto\',\'qa\',\'chat\'):\n        v=met[k];data.append([k,str(v[\'quantidade\']),str(v[\'minimo\']),str(v[\'media_tokens_aprox\']),str(v[\'duplicatas_removidas\'])])\n    tb=Table(data,colWidths=[90,65,65,130,85],repeatRows=1,hAlign=\'LEFT\');tb.setStyle(TableStyle([\n        (\'BACKGROUND\',(0,0),(-1,0),colors.HexColor(\'#183B50\')),(\'TEXTCOLOR\',(0,0),(-1,0),colors.white),(\'ROWBACKGROUNDS\',(0,1),(-1,-1),[colors.white,colors.HexColor(\'#F0F5F7\')]),(\'VALIGN\',(0,0),(-1,-1),\'MIDDLE\'),(\'TOPPADDING\',(0,0),(-1,-1),9),(\'BOTTOMPADDING\',(0,0),(-1,-1),9)]))\n    blocos += [tb,Spacer(1,12),t(f"Recusas estruturais: <b>{met[\'recusas_validas\']}</b> (mínimo: 10). As médias de tokens são <b>estimativas lexicais</b>, não medições com o tokenizador do modelo. Cinco amostras aleatórias de cada corpus foram salvas em <b>amostras_5_por_corpus.json</b>."),\n        h(\'5. Qualidade, riscos e LGPD\'),t(\'O projeto utiliza Qwen2.5-14B-Instruct para auxiliar a geração, com prompt solicitando apoio estrito no texto-fonte, clareza e respeito a quem executa a ação. Filtros automáticos removem entradas inválidas e duplicatas exatas. Os filtros NÃO comprovam fidelidade factual. <b>PENDENTE:</b> equipe revisar pelo menos 20% dos exemplos de IA em revisao_humana_20_porcento.csv e conferir as fontes; corrigir ou excluir exemplos inadequados antes da entrega.\'),\n        h(\'6. Limitações e dificuldades\'),t(\'Desafios observados: respostas repetidas, troca entre responsabilidade do morador e de órgãos públicos, respostas sobre saúde incompletas e divergência entre juiz automático e revisão humana. Exemplos de recusa foram acrescentados para pedidos fora de escopo. Nenhuma taxa de qualidade humana deve ser declarada antes da revisão.\'),\n        h(\'7. Reprodutibilidade e demonstração\'),t(\'Código e arquivos deverão ser disponibilizados em um único link público/compartilhado. <b>PENDENTE:</b> inserir nomes completos da equipe, confirmar execução de ponta a ponta e gravar vídeo de até 5 minutos mostrando upload, processamento, geração dos três JSONL, estatísticas e exemplos.\')]\n    doc.build(blocos)\n    return p\nif __name__==\'__main__\':\n    import sys\n    print(construir(sys.argv[1]))\n',encoding='utf-8')
import fechar_entrega as fechar
importlib.reload(fechar)
existentes=fechar.ler_jsonl(ANTERIOR/'corpus_qa.jsonl')
validos,dups,invalidos=fechar.filtrar(existentes,'qa')
print('QA existentes válidas:',len(validos),'| duplicatas:',dups,'| inválidas:',len(invalidos))
assert len(validos)>=90, 'Quantidade inesperada; não sobrescrever a entrega anterior.'
print('Chat existentes:',len(fechar.ler_jsonl(ANTERIOR/'corpus_chat.jsonl')))
print('Destino corrigido (ainda NÃO criado):',NOVO)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
QA existentes válidas: 95 | duplicatas: 0 | inválidas: 0
Chat existentes: 31
Destino corrigido (ainda NÃO criado): /content/drive/MyDrive/radar-cidadao/entrega_GS_19nov_corrigida


In [25]:
# 2. Segmentação real e modelo Qwen (somente se for necessário gerar exemplos)
import os
os.chdir(PROJETO)
from fabrica_corpus.ingestao import ingerir_pasta
from fabrica_corpus.limpeza import limpar_corpus
from fabrica_corpus.anonimizacao import anonimizar_corpus
from fabrica_corpus.segmentacao import segmentar
from fabrica_corpus import geracao
geracao=importlib.reload(geracao)
assert getattr(geracao,'VERSAO_QA_CHAT',None)=='v8_3', ('É necessário carregar a geração v8.3 antes do complemento. Versão encontrada: '+str(getattr(geracao,'VERSAO_QA_CHAT',None)))
limpos,_=limpar_corpus(ingerir_pasta('dados/entrada'))
anon,_=anonimizar_corpus(limpos)
segs,_=segmentar(anon)
print('Segmentos do projeto:',len(segs))
assert len(segs)>=50
import torch
if len(validos)<100:
    assert torch.cuda.is_available(), 'Ative GPU no Colab (Ambiente de execução > Alterar tipo de ambiente) para rodar Qwen.'
    if 'gerar' not in globals() or not callable(gerar):
        gerar=geracao.carregar_qwen()
CACHE=BASE/'cache_geracao'
CACHE.mkdir(parents=True,exist_ok=True)
print('Versão QA:',geracao.VERSAO_QA_CHAT,'| Cache:',CACHE)

Segmentos do projeto: 306
Versão QA: v8_3 | Cache: /content/drive/MyDrive/radar-cidadao/cache_geracao


In [26]:
# 3. Gerar somente novas QAs, filtrar de novo e exigir mínimo de 100
# Os segmentos já usados pelas 95 QAs são excluídos de preferência.
# Não há inserção artificial de exemplos nem desligamento do juiz.
import copy
qa_ok=list(validos)
segmentos_ja_usados={e.get('segmento_id') for e in qa_ok if e.get('segmento_id')}
candidatos=[s for s in segs if s.id not in segmentos_ja_usados]
print('Segmentos inéditos disponíveis:',len(candidatos))
assert candidatos or len(qa_ok)>=100, 'Sem segmentos inéditos; é preciso revisar a base.'
novos_brutos=[]
for rodada in range(1,5):
    if len(qa_ok)>=100: break
    print(f'\nRODADA {rodada}: {len(qa_ok)}/100 QAs válidas. Gerando candidatas com filtros v8.3...')
    # Seleção diversa com semente fixa, sem mexer nos documentos ou cache.
    pool=list(candidatos)
    random.Random(2026+rodada).shuffle(pool)
    pool=pool[:min(len(pool),90)]
    if not pool: break
    geradas,estat=geracao.gerar_qa(pool,gerar,total=16,por_segmento=2,pasta_cache=CACHE,filtrar_repeticoes=True)
    novos_brutos+=geradas
    normalizadas=fechar.normalizar_qa(geradas)
    combinados,dup,ruins=fechar.filtrar(qa_ok+normalizadas,'qa')
    qa_ok=combinados
    candidatos=[s for s in candidatos if s.id not in {e.get('segmento_id') for e in normalizadas}]
    print('Acumulado válido:',len(qa_ok),'| duplicatas removidas nesta rodada:',dup,'| inválidas:',len(ruins))
    print('Geração:',estat.get('exemplos',0),'aceitas pelo gerador; motivos de descarte internos:',estat.get('motivos_descarte',{}))
assert len(qa_ok)>=100, (f'Geradas apenas {len(qa_ok)} QAs válidas. NÃO exportei pacote incompleto. Reexecute esta célula com GPU e cache disponíveis ou revise as fontes.')
qa_ok=qa_ok[:100]
print('META ALCANÇADA:',len(qa_ok),'QAs únicas e estruturalmente válidas.')

Segmentos inéditos disponíveis: 248

RODADA 1: 95/100 QAs válidas. Gerando candidatas com filtros v8.3...
[perguntas] 1/16 | segmento mcid_2026_mapeamento_riscos_0194 | 1 aprovadas, 3 tentativa(s)
[perguntas] 1/16 | segmento ms_2025_plano_contingencia_chuvas_0025 | 0 aprovadas, 3 tentativa(s)
[perguntas] 2/16 | segmento mcid_2026_mapeamento_riscos_0096 | 1 aprovadas, 3 tentativa(s)
[perguntas] 4/16 | segmento ms_2025_plano_contingencia_chuvas_0035 | 2 aprovadas, 2 tentativa(s)
[perguntas] 6/16 | segmento mcid_2026_mapeamento_riscos_0004 | 2 aprovadas, 3 tentativa(s)
[perguntas] 7/16 | segmento ms_2025_plano_contingencia_chuvas_0041 | 1 aprovadas, 3 tentativa(s)
[perguntas] 8/16 | segmento mcid_2026_mapeamento_riscos_0095 | 1 aprovadas, 3 tentativa(s)
[perguntas] 10/16 | segmento ms_2025_plano_contingencia_chuvas_0074 | 2 aprovadas, 2 tentativa(s)
[perguntas] 12/16 | segmento mcid_2026_mapeamento_riscos_0166 | 2 aprovadas, 2 tentativa(s)
[perguntas] 12/16 | segmento ms_2025_plano_contin

In [27]:
# 4. Exportar pacote CORRIGIDO sem apagar os originais nem fingir revisão humana
import statistics, hashlib, datetime, zipfile, csv
# Protege o trabalho humano prévio: não substitui pasta corrigida se a revisão já foi preenchida.
if NOVO.exists() and (NOVO/'revisao_humana_20_porcento.csv').exists():
    rev=NOVO/'revisao_humana_20_porcento.csv'
    with rev.open(encoding='utf-8-sig',newline='') as f:
        linhas=list(csv.DictReader(f,delimiter=';'))
    if any(r.get('revisor','').strip() or r.get('apoio_na_fonte_S_N','').strip() for r in linhas):
        raise RuntimeError('Há revisão humana preenchida na pasta corrigida. Não sobrescrever: faça backup/reconcilie antes.')
NOVO.mkdir(parents=True,exist_ok=True)
for nome in ('corpus_texto.jsonl','corpus_chat.jsonl'):
    shutil.copy2(ANTERIOR/nome,NOVO/nome)
fechar.salvar_jsonl(NOVO/'corpus_qa.jsonl',qa_ok)
metrics={}
for tipo in ('texto','qa','chat'):
    data=fechar.ler_jsonl(NOVO/f'corpus_{tipo}.jsonl')
    limpa,d,ruins=fechar.filtrar(data,tipo)
    assert len(data)==len(limpa) and len(data)>=fechar.MINIMOS[tipo], f'{tipo} não atingiu a meta'
    anterior_metricas=json.loads((ANTERIOR/'estatisticas.json').read_text(encoding='utf-8'))
    metrics[tipo]={'quantidade':len(data),'minimo':fechar.MINIMOS[tipo], 'quantidade_atingida':True,
      'media_tokens_aprox':round(statistics.mean(fechar.estimar_tokens(e) for e in data),1),
      'duplicatas_removidas':anterior_metricas.get(tipo,{}).get('duplicatas_removidas',0) if tipo!='qa' else anterior_metricas.get('qa',{}).get('duplicatas_removidas',0)+len(novos_brutos)-(len(qa_ok)-len(validos)),
      'invalidos_removidos':anterior_metricas.get(tipo,{}).get('invalidos_removidos',0)}
metrics['recusas_validas']=sum(e.get('tipo')=='recusa' for e in fechar.ler_jsonl(NOVO/'corpus_chat.jsonl'))
metrics['recusas_atingidas']=metrics['recusas_validas']>=10
assert metrics['recusas_atingidas']
metrics.update(revisao_humana_confirmada=False,fontes_documentais_conferidas=False,pronto_para_entrega=False,
 observacao='Estrutura validada; revisão humana >=20%, fidelidade às fontes, README e vídeo pendentes.')
(NOVO/'estatisticas.json').write_text(json.dumps(metrics,ensure_ascii=False,indent=2),encoding='utf-8')
amostras={tipo:random.Random(2026).sample(fechar.ler_jsonl(NOVO/f'corpus_{tipo}.jsonl'),5) for tipo in ('texto','qa','chat')}
(NOVO/'amostras_5_por_corpus.json').write_text(json.dumps(amostras,ensure_ascii=False,indent=2),encoding='utf-8')
fechar.criar_revisao(NOVO)
# Leva as anotações anteriores para a conferência, mas não as marca automaticamente como revisadas.
if (ANTERIOR/'revisao_humana_20_porcento.csv').exists():
    shutil.copy2(ANTERIOR/'revisao_humana_20_porcento.csv',NOVO/'revisao_humana_ANTERIOR_PARA_CONSULTA.csv')
(NOVO/'AVISO_PENDENCIAS.txt').write_text('RASCUNHO: revisar >=20% dos exemplos com fontes originais, conferir formatos e páginas, fechar README e gravar vídeo <=5 minutos. Não declarar entrega aprovada automaticamente.\n',encoding='utf-8')
# Reaproveitar materiais de documentação existentes; sem marcar revisão aprovada.
for nome in ('README_ENTREGA.md','ROTEIRO_VIDEO_5_MIN.md'):
    arquivos=[Path('/content')/nome, PROJETO/nome, ANTERIOR/nome]
    origem=next((p for p in arquivos if p.exists()),None)
    if origem: shutil.copy2(origem,NOVO/nome)
subprocess.run([sys.executable,'-m','pip','-q','install','reportlab'],check=True)
from relatorio_pdf import construir
relatorio=construir(NOVO)
ret=fechar.auditar_pasta(NOVO)
print(json.dumps(ret,ensure_ascii=False,indent=2))
assert all(ret[t]['ok'] for t in ('texto','qa','chat')) and ret['recusas']>=10
ARQUIVO=BASE/'RADAR_CIDADAO_NLP_CORRIGIDO_RASCUNHO.zip'
with zipfile.ZipFile(ARQUIVO,'w',zipfile.ZIP_DEFLATED) as z:
    for arquivo in sorted(NOVO.rglob('*')):
        if arquivo.is_file(): z.write(arquivo,arcname=arquivo.relative_to(BASE))
print('\nPACOTE:',ARQUIVO)
print('REVISÃO HUMANA:',NOVO/'revisao_humana_20_porcento.csv')
print('RELATÓRIO:',relatorio)
print('ATENÇÃO: quantidades corrigidas, mas revisão humana e vídeo continuam pendentes.')

{
  "texto": {
    "quantidade": 306,
    "quantidade_valida": 306,
    "minimo": 50,
    "duplicatas": 0,
    "invalidos": [],
    "ok": true
  },
  "qa": {
    "quantidade": 100,
    "quantidade_valida": 100,
    "minimo": 100,
    "duplicatas": 0,
    "invalidos": [],
    "ok": true
  },
  "chat": {
    "quantidade": 31,
    "quantidade_valida": 31,
    "minimo": 30,
    "duplicatas": 0,
    "invalidos": [],
    "ok": true
  },
  "recusas": 11
}

PACOTE: /content/drive/MyDrive/radar-cidadao/RADAR_CIDADAO_NLP_CORRIGIDO_RASCUNHO.zip
REVISÃO HUMANA: /content/drive/MyDrive/radar-cidadao/entrega_GS_19nov_corrigida/revisao_humana_20_porcento.csv
RELATÓRIO: /content/drive/MyDrive/radar-cidadao/entrega_GS_19nov_corrigida/relatorio_GS_Radar_Cidadao_RASCUNHO.pdf
ATENÇÃO: quantidades corrigidas, mas revisão humana e vídeo continuam pendentes.
